## 1 Load packages, create simulated data

pop.history[0], N=20000

In [361]:
import numpy as np
import pandas as pd
import random
# load et simulation function from simulate_population.py: 
from simulate_population import sim_population

In [ ]:
pop = sim_population(N=20000, step_forward=2, randomseed=42)
pop.step() # move +2y
pop.step() # move + 2y
pop.step() #move + 2y
pop.step() #move + 2y
pop.step() #move + 2y

In [363]:
pop.history[0].head()

,id,start,end,age_start,bmi,hyp,smoke,sex,eth,first_a,...,time_a,time_b,time_c,time_d,time_e,event_a,event_b,event_c,event_d,event_e
0,1,0.0,2,62.1,-1.8,0,0,1,0,NaN,...,5.199,0.059,3.133,142.184,52.815,0,1,0,0,0
1,2,0.0,2,43.0,-0.4,0,0,1,2,NaN,...,4.363,4.903,7.496,100.554,36.235,0,0,0,0,0
2,3,0.0,2,66.9,-1.7,0,0,0,0,NaN,...,15.415,1.925,0.860,31.428,114.841,0,1,1,0,0
3,4,0.0,2,57.7,2.0,1,0,0,2,0.1,...,0.100,0.382,0.095,59.043,89.983,1,1,1,0,0
4,5,0.0,2,23.4,0.2,0,0,1,0,NaN,...,4.573,10.235,15.690,338.610,34.055,0,0,0,0,0


In [395]:
#a single long dataframe - if needed
#long_df = pd.concat(pop.history, ignore_index=True)
#long_df = long_df.sort_values(by=['id', "start"])
#long_df.head(10)

## 2 Create df_long 
	id	age_start	bmi	hyp	smoke	sex	eth	event	time	age
	2	43.0	-0.4	0	0	1	2	first_a	2.172	45.172
	3	66.9	-1.7	0	0	0	0	first_a	2.569	69.469

In [411]:
event_cols = ["first_a", "first_b", "first_c", "first_d", "first_e"]
context_cols = ['age_start', 'bmi', 'hyp', 'smoke', 'sex', 'eth']
df_short = pop.history[5][["id"]+context_cols + event_cols].copy()
df_long = df_short.melt(id_vars = ["id"] + context_cols, 
                       value_vars = event_cols, 
                       var_name = "event", 
                       value_name = "time").dropna(subset=["time"])
df_long["age"] = df_long["age_start"] + df_long["time"]
df_long.head()

,id,age_start,bmi,hyp,smoke,sex,eth,event,time,age
1,2,43.0,-0.4,0,0,1,2,first_a,2.172,45.172
2,3,66.9,-1.7,0,0,0,0,first_a,2.569,69.469
3,4,57.7,2.0,1,0,0,2,first_a,0.100,57.800
5,6,73.6,0.1,0,0,0,0,first_a,1.077,74.677
6,7,61.4,0.2,0,0,1,1,first_a,6.989,68.389


In [412]:
df_short.head()

,id,age_start,bmi,hyp,smoke,sex,eth,first_a,first_b,first_c,first_d,first_e
0,1,62.1,-1.8,0,0,1,0,NaN,0.059,3.118,NaN,NaN
1,2,43.0,-0.4,0,0,1,2,2.172,8.772,NaN,NaN,NaN
2,3,66.9,-1.7,0,0,0,0,2.569,1.925,0.860,NaN,NaN
3,4,57.7,2.0,1,0,0,2,0.100,0.382,0.095,NaN,NaN
4,5,23.4,0.2,0,0,1,0,NaN,NaN,NaN,NaN,NaN


## __2b) Check CoxPH

* eth_beta = np.array([0 if (x==0) else 0.5 if (x==1) else 2 for x in df["eth"]])
* chances are higher if there was a heart failure before 
* eventb_beta = (~df["first_b"].isna()).astype(int) * 0.3
* lp1 = 0.1*np.exp(0.5*df.bmi + 0.7*df.hyp + 0.4*(df.age-50)/15 + eth_beta + eventb_beta)
* time_a = 0.01 + np.round(rng.exponential(1/lp1,len(df)),3)

In [329]:
import pandas as pd
from lifelines import CoxPHFitter
pop = sim_population(N=10000, step_forward=5, randomseed=42)
df_cox = pop.history[0][['end', 'event_b', 'age', 'bmi', 'hyp', 'smoke', 'sex', 'eth']].copy()

In [407]:
cph = CoxPHFitter()
cph.fit(df_cox, duration_col='end', event_col='event_b')
s =cph.summary[['coef', 'se(coef)', 'p']]
round(s,4)

,coef,se(coef),p
covariate,,,
age,0.0267,0.0009,0.0000
bmi,0.0356,0.0152,0.0193
hyp,0.4815,0.0318,0.0000
smoke,-0.0112,0.0388,0.7718
sex,0.0256,0.0278,0.3577
eth,-0.0104,0.0208,0.6163


In [340]:
df1 = pop.history[2][['start','end', 'event_d', 'age', 'bmi', 'hyp', 'smoke', 'sex', 'eth']]
cph.fit(df1, duration_col='end', event_col='event_d', entry_col="start")
s1 =cph.summary[['coef', 'se(coef)', 'p']]
round(s1,4)

,coef,se(coef),p
covariate,,,
age,0.0427,0.0022,0.0000
bmi,-0.0332,0.0319,0.2992
hyp,0.0428,0.0791,0.5879
smoke,-0.1169,0.0925,0.2062
sex,0.0174,0.0642,0.7863
eth,0.1712,0.0458,0.0002


In [410]:
# Now check if going from 0 till the end and taking ever happening event_a returns similar coefficients: 
df1_first = pop.history[2][['first_e','end', 'age', 'bmi', 'hyp', 'smoke', 'sex', 'eth']].copy()
df1_first['event'] = df1_first['first_e'].notna().astype(int) # if a ever happened = 1
df1_first['time'] = df1_first['first_e'].where(df1_first['first_e'].notna(), 10) #time is from first_a column
cph.fit(df1_first[['time','event', 'age', 'bmi', 'hyp', 'smoke', 'sex', 'eth']], 
        duration_col='time', event_col='event')
s1_first = cph.summary[['coef', 'se(coef)', 'p']]
round(s1_first,4)

,coef,se(coef),p
covariate,,,
age,0.0115,0.0016,0.0000
bmi,0.0191,0.0260,0.4633
hyp,0.0262,0.0651,0.6872
smoke,0.0427,0.0721,0.5539
sex,0.0017,0.0524,0.9740
eth,0.1948,0.0368,0.0000


In [409]:
df_short["event_a_1"] =  df_short['first_a'].notna().astype(int)
df_short['time'] = df_short['first_a'].where(df_short['first_a'].notna(), 12)
cph.fit(df_short[['time','event_a_1', 'age_start', 'bmi', 'hyp', 'smoke', 'sex', 'eth']], 
        duration_col='time', event_col='event_a_1')
sdfg_first = cph.summary[['coef', 'se(coef)', 'p']]
round(sdfg_first,4)

,coef,se(coef),p
covariate,,,
age_start,0.0266,0.0005,0.0000
bmi,0.4857,0.0084,0.0000
hyp,0.7053,0.0190,0.0000
smoke,-0.0328,0.0218,0.1333
sex,0.0092,0.0156,0.5551
eth,0.7885,0.0128,0.0000


## 3 Fit transformer to the simulated population

Embeddings: 1) Static context (time-invariant)   sex, eth; 2) Dynamic context (time-varying, per step)  age, bmi, hyp, smoke, 3) Token per step  {NONE, A, B, C, D, E}

Encodings (to sum together): 1) Event token embedding; 2) Dynamic covariate embedding; 3) Static covariate embedding (broadcast over time)

### 3a) Shape df_long for transformer

In [417]:
def build_event_sequences(df_long):
    sequences = []
    for pid, g in df_long.groupby("id"):
        g = g.sort_values("time")
        tokens = g["event"].str.replace("first_", "", regex=False).tolist()
        features = g[["age", "bmi", "hyp", "smoke", "sex", "eth"]].to_numpy()
        times = g["time"].to_numpy()
        sequences.append({ "id": pid, "tokens": tokens, "features": features,"times": times })
    return sequences

In [420]:
dfdf = build_event_sequences(df_long)

In [426]:
dfdf[1]

{'id': 2,
 'tokens': ['a', 'b'],
 'features': array([[45.172, -0.4  ,  0.   ,  0.   ,  1.   ,  2.   ],
        [51.772, -0.4  ,  0.   ,  0.   ,  1.   ,  2.   ]]),
 'times': array([2.172, 8.772])}

In [427]:
class EventSequenceTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=64, nhead=4):
        super().__init__()

        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.cov_emb = nn.Linear(6, d_model)    # age, bmi, hyp, smoke, sex, eth
        self.time_emb = nn.Linear(1, d_model)   # Δt
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, batch_first=True )
        
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.out = nn.Linear(d_model, vocab_size)
    
    def forward(self, token_ids, covariates, dt):
        x = self.token_emb(token_ids) + self.cov_emb(covariates) + self.time_emb(dt)
        x = self.encoder(x)
        return self.out(x)


### 3b) Training: objective (correct for this data) - next-event prediction
P(event_k | event_<k, covariates_<k)

In [436]:
def pad_sequences(sequences, vocab):
    token_seqs = []
    feat_seqs = []
    dt_seqs = []
    lengths = []

    for s in sequences:
        tokens = [vocab[t] for t in s["tokens"]]
        feats = s["features"]
        times = s["times"]
        dt = np.diff(np.concatenate([[0.0], times]))
        token_seqs.append(tokens)
        feat_seqs.append(feats)
        dt_seqs.append(dt[:, None])  # shape (T,1)
        lengths.append(len(tokens))

    max_len = max(lengths)

    def pad(x, value=0): return np.array( [xi + [value] * (max_len - len(xi)) for xi in x] )

    X_tokens = torch.tensor(pad(token_seqs), dtype=torch.long)
    X_feats = torch.tensor(
        np.stack([np.vstack([f, np.zeros((max_len - len(f), f.shape[1]))]) for f in feat_seqs ]),dtype=torch.float32)
    X_dt = torch.tensor(
        np.stack([np.vstack([d, np.zeros((max_len - len(d), 1))]) for d in dt_seqs]),dtype=torch.float32)

    lengths = torch.tensor(lengths)

    return X_tokens, X_feats, X_dt, lengths


In [ ]:
#STANDARD CROSS_ENTROPY LOSS

def masked_cross_entropy(logits, targets, lengths):
    """     logits:  [B, T-1, V]     targets: [B, T-1]     lengths: [B]   """
    B, T, V = logits.shape
    mask = ( torch.arange(T)[None, :] < (lengths - 1)[:, None])
    loss = torch.nn.functional.cross_entropy(
        logits.reshape(-1, V), targets.reshape(-1),  reduction="none" )
    loss = loss.view(B, T)
    
    return (loss * mask).sum() / mask.sum()

In [449]:
# ALTERNATIVE DELPHI-2M style loss

def masked_competing_exp_loss(logits, targets, dt, lengths, alpha=1.0):
    """
    logits:   [B, T, K]
    targets:  [B, T]
    dt:       [B, T]
    lengths:  [B]
    """
    B, T, K = logits.shape
    device = logits.device

    targets = targets.long()

    mask = (
        torch.arange(T, device=device)[None, :]
        < lengths[:, None]
    )

    # event loss
    loss_event = torch.nn.functional.cross_entropy(
        logits.reshape(-1, K),
        targets.reshape(-1),
        reduction="none"
    ).view(B, T)

    # time loss
    log_lambda_sum = torch.logsumexp(logits, dim=-1)
    lambda_sum = torch.exp(log_lambda_sum)
    loss_time = -log_lambda_sum + lambda_sum * dt

    loss = loss_event + alpha * loss_time
    return (loss * mask).sum() / mask.sum()


In [454]:
def train_transformer(model, sequences, vocab, n_epochs=10,
                       lr=1e-3, batch_size=32, alpha = 1.0, device="cpu" ):
    model.to(device)
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    # build tensors once
    X_tokens, X_feats, X_dt, lengths = pad_sequences(sequences, vocab)
    X_tokens = X_tokens.to(device)
    X_feats = X_feats.to(device)
    X_dt = X_dt.to(device)
    lengths = lengths.to(device)

    N = X_tokens.size(0)
    for epoch in range(n_epochs):
        perm = torch.randperm(N)
        total_loss = 0.0
        for i in range(0, N, batch_size):
            idx = perm[i:i+batch_size]
            tokens = X_tokens[idx]
            feats = X_feats[idx]
            dt = X_dt[idx]
            lens = lengths[idx]
            optimizer.zero_grad()
            # predict hazards for the next event
            logits = model(tokens[:, :-1],  feats[:, :-1],  dt[:, :-1] ) #history, covariates then, previous gap
            # compute the loss

            loss = masked_competing_exp_loss(
                logits=logits,
                targets=tokens[:, 1:],   # next event
                dt=dt[:, 1:].squeeze(-1),             # waiting time to next event
                lengths=lens - 1,
                alpha=alpha
            )
            ## alternative (no hazard, just cross-entropy):
            ## loss = masked_cross_entropy( logits, tokens[:, 1:], lens)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(idx)

        avg_loss = total_loss / N
        print(f"Epoch {epoch+1:03d} | loss = {avg_loss:.4f}")


In [455]:
vocab = {"a": 0, "b": 1, "c": 2, "d": 3, "e": 4}
model = EventSequenceTransformer(vocab_size=len(vocab), d_model=64,  nhead=4)
train_transformer(model, dfdf, vocab, n_epochs=20, lr=1e-3, batch_size=64)


Epoch 001 | loss = 3.1810
Epoch 002 | loss = 2.6866
Epoch 003 | loss = 2.5263
Epoch 004 | loss = 2.4307
Epoch 005 | loss = 2.3791
Epoch 006 | loss = 2.3367
Epoch 007 | loss = 2.3102
Epoch 008 | loss = 2.2768
Epoch 009 | loss = 2.2433
Epoch 010 | loss = 2.2253
Epoch 011 | loss = 2.1954
Epoch 012 | loss = 2.1488
Epoch 013 | loss = 2.1127
Epoch 014 | loss = 2.0614
Epoch 015 | loss = 2.0595
Epoch 016 | loss = 2.0142
Epoch 017 | loss = 2.0129
Epoch 018 | loss = 2.0025
Epoch 019 | loss = 1.9723
Epoch 020 | loss = 1.9766
